# 🪞 Reflection Agent in LangGraph

#### https://www.youtube.com/watch?v=L7J5Etn23iU&list=PLNIQLFWpQMRXmns-7UarmPIR6DN7bgEzZ&index=8

<table>
  <tr>
    <td width="20%">
      <img src="../images/Reflection Agent Pattern.png" alt="Description Image" style="width:100%;">
    </td>
    <td width="20%">
      <img src="../images/Reflection Agent Pattern-2.png" alt="Description Image" style="width:100%;">
    </td>
    <td width="60%" style="vertical-align: top;">
      <p> A basic reflection agent system consists of </p>
      <ul>
      <li> A Generator Agent </li>
      <li> A Reflection Agent </li>
      </ul>
    </td>
  </tr>
</table>


In [13]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from dotenv import load_dotenv
from langgraph.graph import START, END, StateGraph

In [8]:
load_dotenv()

True

In [5]:
GOOGLE_MODEL="gemini-2.5-flash"
OPENAI_MODEL="gpt-5-nano"

In [2]:
generation_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a twitter techie influencer assistant tasked with writing excellent twitter posts."
            " Generate the best twitter post possible for the user's request."
            " If the user provides critique, respond with a revised version of your previous attempts.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

In [3]:
reflection_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a viral twitter influencer grading a tweet. Generate critique and recommendations for the user's tweet."
            "Always provide detailed recommendations, including requests for length, virality, style, etc.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)


In [6]:
llm_generator = ChatGoogleGenerativeAI(model=GOOGLE_MODEL)
llm_reflector = ChatOpenAI(model=OPENAI_MODEL, temperature=0.7)

In [7]:
generation_chain = generation_prompt | llm_generator
reflection_chain = reflection_prompt | llm_reflector

In [ ]:
from typing import Annotated, TypedDict
from dotenv import load_dotenv
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph import END, StateGraph, START
from langgraph.graph.message import add_messages
from chains import generation_chain, reflection_chain

load_dotenv()

# --- 1. Define the State ---
class AgentState(TypedDict):
    # Annotated with add_messages so new nodes append to the list
    messages: Annotated[list[BaseMessage], add_messages]

REFLECT = "reflect"
GENERATE = "generate"

# --- 2. Define Node Logic ---
def generate_node(state: AgentState):
    # Pass the message list to your chain
    response = generation_chain.invoke({"messages": state["messages"]})
    # Return the key matching our State TypedDict
    return {"messages": [response]}

def reflect_node(state: AgentState):
    # Use the current history for reflection
    response = reflection_chain.invoke({"messages": state["messages"]})
    # Wrap reflection in a HumanMessage to "prod" the generator
    return {"messages": [HumanMessage(content=response.content)]}

# --- 3. Build the Graph ---
builder = StateGraph(AgentState)

builder.add_node(GENERATE, generate_node)
builder.add_node(REFLECT, reflect_node)

builder.add_edge(START, GENERATE)

def should_continue(state: AgentState):
    # In StateGraph, you access the messages via the state key
    if len(state["messages"]) > 6:
        return END
    return REFLECT

builder.add_conditional_edges(GENERATE, should_continue)
builder.add_edge(REFLECT, GENERATE)

# --- 4. Compile and Run ---
app = builder.compile()

# Visualizing
print(app.get_graph().draw_mermaid())

# Invocation
# Note: Input must now be a dictionary matching the AgentState schema
initial_input = {"messages": [HumanMessage(content="AI Agents taking over content creation")]}
response = app.invoke(initial_input)

# Output the final list of messages
for message in response["messages"]:
    print(f"--- {type(message).__name__} ---\n{message.content}\n")
